# UF Policy Assistant — Setup and data check

This notebook verifies required packages, loads the prepared policy dataset, prints basic dataset summaries, and checks that the local `policies` ChromaDB collection is available. It does not call the language-model API and does not require an API key.

In [1]:
from pathlib import Path
import importlib.metadata
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src").is_dir() and (path / "data").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not find the project root. Open this notebook from the project directory.")

DATASET_FILE = PROJECT_ROOT / "data" / "raw" / "Web_Scrapped_Dataset" / "policies_data.xlsx"
CHROMA_DIR = PROJECT_ROOT / "data" / "chroma_db"
if not DATASET_FILE.is_file():
    raise FileNotFoundError(f"Prepared policy workbook not found: {DATASET_FILE}")

required_packages = ["pandas", "openpyxl", "chromadb", "ipykernel"]
application_packages = ["sentence-transformers", "openai", "streamlit"]
package_versions = {}
missing_packages = []
for package in required_packages + application_packages:
    try:
        package_versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        if package in required_packages:
            missing_packages.append(package)

print(f"Python: {sys.version.split()[0]}")
print(f"Project root: {PROJECT_ROOT}")
print("Dependency versions:")
for package, version in package_versions.items():
    print(f"  {package}: {version}")
if missing_packages:
    raise RuntimeError(f"Missing packages: {', '.join(missing_packages)}. Install requirements.txt in this environment.")
missing_application_packages = [package for package in application_packages if package not in package_versions]
if missing_application_packages:
    print(f"Application dependencies missing from this kernel: {', '.join(missing_application_packages)}")
    print("Select the project .venv kernel or install requirements.txt there before running the RAG app.")
else:
    print("Application dependencies are installed in this kernel.")
print("Notebook and data environment check passed.")

Python: 3.11.16
Project root: /Users/rohan/Desktop/term-project-group_7
Dependency versions:
  pandas: 2.3.3
  openpyxl: 3.1.5
  chromadb: 1.2.1
  ipykernel: 7.0.1
  sentence-transformers: 5.1.1
  openai: 2.6.0
Application dependencies missing from this kernel: streamlit
Select the project .venv kernel or install requirements.txt there before running the RAG app.
Notebook and data environment check passed.


In [2]:
import pandas as pd
from IPython.display import display

policies = pd.read_excel(DATASET_FILE, sheet_name="Policies")
required_columns = {"title", "category", "type", "link", "content"}
missing_columns = required_columns.difference(policies.columns)
if missing_columns:
    raise ValueError(f"Policy workbook is missing columns: {sorted(missing_columns)}")
if policies.empty:
    raise ValueError("The policy workbook loaded successfully but has no rows.")

print(f"Loaded: {DATASET_FILE.relative_to(PROJECT_ROOT)}")
print(f"Rows (policy passages): {len(policies):,}")
print(f"Columns: {len(policies.columns)}")
print(f"Unique policy titles: {policies['title'].nunique():,}")
print(f"Rows with non-empty passage text: {policies['content'].fillna('').str.strip().ne('').sum():,}")

print("\nPassages by document type:")
display(policies["type"].fillna("Unknown").value_counts().rename_axis("type").to_frame("passages"))

print("Sample records (passage text shortened):")
preview_columns = ["title", "category", "type", "link", "chunk_number", "content"]
preview = policies[[column for column in preview_columns if column in policies.columns]].head(5).copy()
preview["content"] = preview["content"].fillna("").str.slice(0, 250)
display(preview)

Loaded: data/raw/Web_Scrapped_Dataset/policies_data.xlsx
Rows (policy passages): 1,887
Columns: 7
Unique policy titles: 189
Rows with non-empty passage text: 1,887

Passages by document type:


,passages
type,
Policy,946
Regulation,941


Sample records (passage text shortened):


,title,category,type,link,chunk_number,content
0,Title IX,Compliance and Ethics,Policy,https://policy.ufl.edu/policy/title-ix/,1,Title IX – Regulation and Policy Hub Purpose A...
1,Title IX,Compliance and Ethics,Policy,https://policy.ufl.edu/policy/title-ix/,2,. 2. Applicability This Policy applies to the ...
2,Title IX,Compliance and Ethics,Policy,https://policy.ufl.edu/policy/title-ix/,3,". This may include, but is not limited to, con..."
3,Title IX,Compliance and Ethics,Policy,https://policy.ufl.edu/policy/title-ix/,4,. Sex discrimination other than Prohibited Con...
4,Title IX,Compliance and Ethics,Policy,https://policy.ufl.edu/policy/title-ix/,5,. Delegees will be trained and supervised by U...


In [3]:
import chromadb

if not CHROMA_DIR.is_dir():
    raise FileNotFoundError(f"Local ChromaDB directory not found: {CHROMA_DIR}")

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
policy_collection = chroma_client.get_collection(name="policies")
print(f"ChromaDB path: {CHROMA_DIR.relative_to(PROJECT_ROOT)}")
print(f"Collection: {policy_collection.name}")
print(f"Indexed passages: {policy_collection.count():,}")
assert policy_collection.count() > 0, "The policies collection is empty."
print("Setup and data checks passed.")

ChromaDB path: data/chroma_db
Collection: policies
Indexed passages: 1,887
Setup and data checks passed.
